# 📊 Fundamental Analysis Report
## 미국 주식 투자전략 — 펀더멘탈 분석 보고서

| 셀 | 역할 |
|----|---------|
| 1  | 경로 자동 감지 (노트북 / 데스크탑) |
| 2  | Import & DB 연결 |
| 3  | 파라미터 설정 |
| 4  | DB 데이터 로드 (FCFF / Relative / Revenue) |
| 5  | 종합 Valuation 테이블 계산 |
| 6  | 수출입 동향 분석 (수혜 업종 추정) |
| 7  | 매출 고성장 기업 추출 |
| 8  | 교집합 필터 + AI Pick |
| 9  | 가중 적정주가 & Upside 순위 (매수 후보) |
| 10 | 수출입 Top-20 전월/당월 예측 변화 |
| 11 | Excel 보고서 저장 |


## Cell 1 · 경로 자동 감지

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ── 후보 루트 (노트북 / 데스크탑) ────────────────────────────
_CANDIDATE_ROOTS = [
    r'C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast',
    r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy',
]

def _setup_path() -> str:
    """현재 파일 위치에서 DATA 폴더를 가진 루트를 탐색 → sys.path 등록"""
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    # 상위 폴더 탐색
    for p in [start] + list(start.parents):
        if (p / 'DATA').is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f'[PATH] 자동 감지 성공 : {root}')
            return root
    # 후보 경로 직접 시도
    for candidate in _CANDIDATE_ROOTS:
        data_dir = os.path.join(candidate, 'DATA')
        if os.path.isdir(candidate) and os.path.isdir(data_dir):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f'[PATH] 후보 경로 사용 : {candidate}')
            return candidate
    raise EnvironmentError(
        'DATA 폴더를 찾을 수 없습니다.\n'
        '_CANDIDATE_ROOTS 를 현재 환경에 맞게 수정하세요.'
    )

_ROOT = _setup_path()
print(f'[확인] 프로젝트 루트 : {_ROOT}')
print(f'[확인] DATA 경로    : {os.path.join(_ROOT, "DATA")}')

## Cell 2 · Import & DB 연결

In [ ]:
import gc, math, warnings
from datetime import datetime, timedelta
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import pymysql
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display
from sqlalchemy import text

matplotlib.rcParams['font.family']       = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 내부 모듈 ────────────────────────────────────────────────
from DATA.config import get_db_info, get_engine

# ── Screened Ticker List import ──────────────────────────────
# DATA 폴더 안에 us_target_ticker_list_screened_20260411.py 가 있어야 함
try:
    from DATA.us_target_ticker_list_screened_20260411 import (
        ticker_list        as US_TICKER_LIST,
        ticker_sector_map  as US_SECTOR_MAP,
    )
    print(f'[OK] Ticker 리스트 로드 : {len(US_TICKER_LIST)}개')
except ImportError:
    # 파일명이 다를 경우 DATA 폴더에서 자동 탐색
    import glob
    candidates = glob.glob(os.path.join(_ROOT, 'DATA', 'us_target_ticker_list_screened*.py'))
    if candidates:
        import importlib.util
        spec = importlib.util.spec_from_file_location('ticker_mod', candidates[-1])
        mod  = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        US_TICKER_LIST = mod.ticker_list
        US_SECTOR_MAP  = getattr(mod, 'ticker_sector_map', {})
        print(f'[OK] Ticker 리스트 로드 (자동탐색) : {len(US_TICKER_LIST)}개')
    else:
        raise FileNotFoundError('us_target_ticker_list_screened_*.py 파일을 DATA 폴더에 넣어주세요.')

# ── DB 연결 ──────────────────────────────────────────────────
db_info = get_db_info()
engine  = get_engine(db_info)

def get_conn():
    return pymysql.connect(
        host       = db_info['host'],
        port       = int(db_info.get('port', 3307)),
        user       = db_info['user'],
        password   = db_info['password'],
        db         = db_info.get('database', 'investar'),
        charset    = 'utf8mb4',
        autocommit = False,
        cursorclass= pymysql.cursors.DictCursor,
    )

try:
    with engine.connect() as c:
        c.execute(text('SELECT 1'))
    print(f'[OK] DB 연결 성공  host={db_info["host"]}:{db_info["port"]}')
except Exception as e:
    print(f'[FAIL] DB 연결 실패: {e}')

## Cell 3 · 파라미터 설정  ← 여기만 수정

In [ ]:
# ══════════════════════════════════════════════════════════════
#  분석 파라미터 — 필요에 따라 수정하세요
# ══════════════════════════════════════════════════════════════

EVAL_DATE        = datetime.today().strftime('%Y-%m-%d')   # 평가 기준일 (오늘)
REVENUE_MODEL    = 'Ensemble'   # SARIMA / ETS / Prophet / LSTM / Theta / Ensemble
REVENUE_ITEM     = 'sale'

# ── Valuation 가중치 ─────────────────────────────────────────
W_FCFF  = 0.60    # FCFF DCF 비중
W_RELV  = 0.40    # Relative Valuation 비중
#  (RIM 모형 결과가 있으면 W_RIM 추가 가능)

# ── 매수 후보 Upside 최소 기준 ───────────────────────────────
UPSIDE_LARGE_CAP = 0.30   # 대형주 (시총 > 10B) : 30%
UPSIDE_SMALL_CAP = 0.40   # 중소형주 (시총 ≤ 10B): 40%
SMALL_CAP_THRESH = 10.0   # 중소형 기준 (단위 B USD)

# ── 매출 고성장 기준 ─────────────────────────────────────────
GROWTH_4Q_THRESH = 15.0   # 4Q 매출 성장률 기준 (%)

# ── DB 테이블명 ──────────────────────────────────────────────
TBL_REVENUE    = 'us_revenue_forecast_data'         # 매출 예측
TBL_FCFF       = 'us_fcff_dcf_valuation'            # FCFF DCF
TBL_RELV       = 'us_relative_valuation'            # Relative Valuation
TBL_KR_TRADE   = 'korea_monthly_trade_data'         # 한국 수출 실적
TBL_KR_FCST    = 'korea_trade_forecast'             # 한국 수출 예측 (존재 시)
TBL_US_EXP_M   = 'us_trade_export_monthly_with_forecast'   # 미국 수출 예측(월)
TBL_US_EXP_Q   = 'us_trade_export_quarter_with_forecast'   # 미국 수출 예측(분기)
TBL_US_IMP_M   = 'us_trade_import_monthly_with_forecast'   # 미국 수입 예측(월)
TBL_US_IMP_Q   = 'us_trade_import_quarter_with_forecast'   # 미국 수입 예측(분기)

# ── 결과 Excel 저장 경로 ─────────────────────────────────────
REPORT_DIR  = os.path.join(_ROOT, 'reports')
os.makedirs(REPORT_DIR, exist_ok=True)
REPORT_FILE = os.path.join(REPORT_DIR, f'fundamental_report_{EVAL_DATE}.xlsx')

print(f'[OK] 파라미터 설정 완료')
print(f'     평가일     : {EVAL_DATE}')
print(f'     FCFF 비중  : {W_FCFF*100:.0f}%  /  Relative 비중 : {W_RELV*100:.0f}%')
print(f'     리포트 경로: {REPORT_FILE}')

## Cell 4 · DB 데이터 로드

In [ ]:
# ──────────────────────────────────────────────────────────────
# 4-1. FCFF DCF 최신 결과 로드
# ──────────────────────────────────────────────────────────────
def load_fcff(tickers: List[str]) -> pd.DataFrame:
    """us_fcff_dcf_valuation 에서 ticker별 최신 date 레코드 추출"""
    ticker_str = "','".join(tickers)
    sql = f"""
        SELECT f.*
        FROM {TBL_FCFF} f
        INNER JOIN (
            SELECT ticker, MAX(date) AS max_date
            FROM {TBL_FCFF}
            WHERE ticker IN ('{ticker_str}')
            GROUP BY ticker
        ) m ON f.ticker = m.ticker AND f.date = m.max_date
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            rows = cur.fetchall()
        return pd.DataFrame(rows)
    finally:
        conn.close()

# ──────────────────────────────────────────────────────────────
# 4-2. Relative Valuation 최신 결과 로드
# ──────────────────────────────────────────────────────────────
def load_relv(tickers: List[str]) -> pd.DataFrame:
    """us_relative_valuation 에서 ticker별 최신 date 레코드 추출"""
    ticker_str = "','".join(tickers)
    sql = f"""
        SELECT r.*
        FROM {TBL_RELV} r
        INNER JOIN (
            SELECT ticker, MAX(date) AS max_date
            FROM {TBL_RELV}
            WHERE ticker IN ('{ticker_str}')
            GROUP BY ticker
        ) m ON r.ticker = m.ticker AND r.date = m.max_date
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            rows = cur.fetchall()
        return pd.DataFrame(rows)
    finally:
        conn.close()

# ──────────────────────────────────────────────────────────────
# 4-3. Revenue Forecast 4Q 성장률 로드 (전 ticker 일괄)
# ──────────────────────────────────────────────────────────────
def load_revenue_growth(tickers: List[str], model: str = REVENUE_MODEL) -> pd.DataFrame:
    """
    us_revenue_forecast_data 에서 각 ticker의
    - 실제 최근 4Q 합산 vs 예측 향후 4Q 합산 → growth_4q
    - 실제 최근 8Q 합산 vs 예측 향후 8Q 합산 → growth_8q
    """
    ticker_str = "','".join(tickers)
    # 최신 forecast_date 기준으로 actual + forecast 일괄 조회
    sql_act = f"""
        SELECT a.ticker, a.date, a.value
        FROM {TBL_REVENUE} a
        INNER JOIN (
            SELECT ticker, MAX(forecast_date) AS fd
            FROM {TBL_REVENUE}
            WHERE ticker IN ('{ticker_str}') AND item='{REVENUE_ITEM}'
            GROUP BY ticker
        ) m ON a.ticker=m.ticker AND a.forecast_date=m.fd
        WHERE a.ticker IN ('{ticker_str}')
          AND a.item='{REVENUE_ITEM}'
          AND a.model='actual'
        ORDER BY a.ticker, a.date
    """
    sql_fct = f"""
        SELECT a.ticker, a.date, a.value
        FROM {TBL_REVENUE} a
        INNER JOIN (
            SELECT ticker, MAX(forecast_date) AS fd
            FROM {TBL_REVENUE}
            WHERE ticker IN ('{ticker_str}') AND item='{REVENUE_ITEM}'
            GROUP BY ticker
        ) m ON a.ticker=m.ticker AND a.forecast_date=m.fd
        WHERE a.ticker IN ('{ticker_str}')
          AND a.item='{REVENUE_ITEM}'
          AND a.model='{model}'
          AND a.data_type='forecast'
        ORDER BY a.ticker, a.date
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql_act)
            act_rows = cur.fetchall()
            cur.execute(sql_fct)
            fct_rows = cur.fetchall()
    finally:
        conn.close()

    act_df = pd.DataFrame(act_rows)
    fct_df = pd.DataFrame(fct_rows)
    if act_df.empty or fct_df.empty:
        return pd.DataFrame(columns=['ticker','growth_4q','growth_8q'])

    records = []
    for tkr in tickers:
        a = act_df[act_df['ticker']==tkr].sort_values('date')['value'].tolist()
        f = fct_df[fct_df['ticker']==tkr].sort_values('date')['value'].tolist()
        if len(a) < 4 or len(f) < 4:
            continue
        base4  = sum(a[-4:]);
        g4     = (sum(f[:4]) / base4 - 1) * 100 if base4 else None
        base8  = sum(a[-8:]) if len(a) >= 8 else None
        g8     = (sum(f[:8]) / base8 - 1) * 100 if (base8 and len(f) >= 8) else None
        records.append({'ticker': tkr, 'growth_4q': g4, 'growth_8q': g8})
    return pd.DataFrame(records)

# ── 실행 ─────────────────────────────────────────────────────
print('데이터 로드 중...')
df_fcff  = load_fcff(US_TICKER_LIST)
df_relv  = load_relv(US_TICKER_LIST)
df_rev   = load_revenue_growth(US_TICKER_LIST)

print(f'[FCFF]     {len(df_fcff)}개 ticker')
print(f'[Relative] {len(df_relv)}개 ticker')
print(f'[Revenue]  {len(df_rev)}개 ticker')

## Cell 5 · 종합 Valuation 테이블 계산

In [ ]:
# ──────────────────────────────────────────────────────────────
# 5-1. FCFF + Relative 가중 적정주가 계산
# ──────────────────────────────────────────────────────────────

# FCFF 핵심 컬럼 선택
fcff_key = ['ticker','date','target_price','current_price','upside_pct',
             'roic','discount_rate','g_terminal','enterprise_value',
             'equity_value','shares','tax_rate','opm_forecast',
             'reinvestment_rate','fcff']
fcff_key = [c for c in fcff_key if c in df_fcff.columns]
df_f = df_fcff[fcff_key].copy() if not df_fcff.empty else pd.DataFrame()
if not df_f.empty:
    df_f = df_f.rename(columns={
        'target_price'  : 'tp_fcff',
        'current_price' : 'price_fcff',
        'upside_pct'    : 'upside_fcff',
        'date'          : 'date_fcff',
    })

# Relative 핵심 컬럼 선택
relv_key = ['ticker','date','tp_avg','current_price','upside_avg',
             'roe_y1','roe_y2','g_est','re_mid','beta_ensemble',
             'pbr_theory','psr_theory','per_theory',
             'tp_pbr','tp_psr','tp_per','sector']
relv_key = [c for c in relv_key if c in df_relv.columns]
df_r = df_relv[relv_key].copy() if not df_relv.empty else pd.DataFrame()
if not df_r.empty:
    df_r = df_r.rename(columns={
        'tp_avg'        : 'tp_relv',
        'current_price' : 'price_relv',
        'upside_avg'    : 'upside_relv',
        'date'          : 'date_relv',
    })

# 합치기
if not df_f.empty and not df_r.empty:
    df_val = pd.merge(df_f, df_r, on='ticker', how='outer')
elif not df_f.empty:
    df_val = df_f.copy()
elif not df_r.empty:
    df_val = df_r.copy()
else:
    df_val = pd.DataFrame(columns=['ticker'])

# 현재 주가 통합 (FCFF 우선)
if 'price_fcff' in df_val.columns and 'price_relv' in df_val.columns:
    df_val['current_price'] = df_val['price_fcff'].fillna(df_val['price_relv'])
elif 'price_fcff' in df_val.columns:
    df_val['current_price'] = df_val['price_fcff']
elif 'price_relv' in df_val.columns:
    df_val['current_price'] = df_val['price_relv']

# 가중 적정주가
def weighted_tp(row):
    tp_f = row.get('tp_fcff') if pd.notna(row.get('tp_fcff', float('nan'))) else None
    tp_r = row.get('tp_relv') if pd.notna(row.get('tp_relv', float('nan'))) else None
    if tp_f is not None and tp_r is not None:
        return tp_f * W_FCFF + tp_r * W_RELV
    elif tp_f is not None:
        return tp_f
    elif tp_r is not None:
        return tp_r
    return None

df_val['tp_weighted'] = df_val.apply(weighted_tp, axis=1)
df_val['upside_weighted'] = (
    (df_val['tp_weighted'] - df_val['current_price']) / df_val['current_price'] * 100
)

# 섹터/시가총액 정보 보충
df_val['sector'] = df_val.apply(
    lambda r: US_SECTOR_MAP.get(r['ticker'], {}).get('sector', r.get('sector', 'N/A')), axis=1
)
df_val['mktCap_B'] = df_val['ticker'].map(
    lambda t: US_SECTOR_MAP.get(t, {}).get('mktCap_B', None)
)

# Revenue 성장률 합치기
if not df_rev.empty:
    df_val = pd.merge(df_val, df_rev[['ticker','growth_4q','growth_8q']], on='ticker', how='left')

print(f'[OK] Valuation 테이블 완성: {len(df_val)}개 ticker')
display(df_val[['ticker','sector','current_price','tp_fcff','tp_relv',
                'tp_weighted','upside_weighted','growth_4q']].head(20))

## Cell 6 · 수출입 동향 분석 — 수혜 업종 추정

In [ ]:
# ──────────────────────────────────────────────────────────────
# HS Code → 수혜 업종(섹터) 매핑 테이블
# (대표 HS Code와 미국 주식 섹터 간 연결)
# ──────────────────────────────────────────────────────────────
HS_SECTOR_MAP = {
    # 반도체/전자
    '854231': 'Technology', '854232': 'Technology', '854239': 'Technology',
    '854290': 'Technology', '854210': 'Technology',
    '854110': 'Technology', '854140': 'Technology',
    # 컴퓨터/통신장비
    '847130': 'Technology', '847141': 'Technology', '847150': 'Technology',
    '851762': 'Technology', '851769': 'Technology',
    # 자동차/부품
    '870323': 'Consumer Cyclical', '870324': 'Consumer Cyclical',
    '870840': 'Consumer Cyclical', '870850': 'Consumer Cyclical',
    # 항공우주/방산
    '880240': 'Industrials', '880260': 'Industrials',
    '890110': 'Industrials',
    # 의약품/바이오
    '300490': 'Healthcare', '300390': 'Healthcare',
    '382200': 'Healthcare',
    # 기계/산업재
    '841381': 'Industrials', '841451': 'Industrials',
    '847989': 'Industrials',
    # 화학
    '390110': 'Basic Materials', '390120': 'Basic Materials',
    '290511': 'Basic Materials',
    # 농산물/식품
    '100190': 'Consumer Defensive', '120190': 'Consumer Defensive',
    # 에너지(참고용)
    '270900': 'Energy', '271012': 'Energy',
}

# ──────────────────────────────────────────────────────────────
# 미국 수출 예측 테이블에서 증감률 계산
# ──────────────────────────────────────────────────────────────
def load_us_export_growth() -> pd.DataFrame:
    """
    us_trade_export_monthly_with_forecast 테이블에서
    최신 created_at 기준 향후 4분기 합산 vs 실제 직전 4분기 합산
    """
    try:
        conn = get_conn()
        with conn.cursor() as cur:
            # 최신 예측 날짜 확인
            cur.execute(f'SELECT MAX(created_at) AS max_date FROM {TBL_US_EXP_M}')
            row = cur.fetchone()
            if not row or not row['max_date']:
                print(f'[WARN] {TBL_US_EXP_M} 테이블 데이터 없음')
                return pd.DataFrame()
            latest_created = row['max_date']

            # 실제값 (is_forecast=0)
            cur.execute(f"""
                SELECT hs_code, date_month_end AS date, expDlr AS value
                FROM {TBL_US_EXP_M}
                WHERE is_forecast=0
                ORDER BY hs_code, date_month_end
            """)
            act_rows = cur.fetchall()

            # 예측값 (is_forecast=1, 최신 created_at)
            cur.execute(f"""
                SELECT hs_code, date_month_end AS date, expDlr_forecast AS value
                FROM {TBL_US_EXP_M}
                WHERE is_forecast=1 AND created_at='{latest_created}'
                ORDER BY hs_code, date_month_end
            """)
            fct_rows = cur.fetchall()
        conn.close()

        act_df = pd.DataFrame(act_rows)
        fct_df = pd.DataFrame(fct_rows)
        if act_df.empty or fct_df.empty:
            return pd.DataFrame()

        records = []
        for code in act_df['hs_code'].unique():
            a = act_df[act_df['hs_code']==code].sort_values('date')['value'].tolist()
            f = fct_df[fct_df['hs_code']==code].sort_values('date')['value'].tolist()
            if len(a) < 4 or len(f) < 4:
                continue
            base4 = sum(a[-4:])
            g4    = (sum(f[:4]) / base4 - 1) * 100 if base4 else None
            records.append({
                'hs_code': str(code)[:6],
                'growth_4q': g4,
                'recent_avg': sum(a[-3:]) / 3 if len(a) >= 3 else None,
            })
        return pd.DataFrame(records)
    except Exception as e:
        print(f'[WARN] 미국 수출 데이터 로드 실패: {e}')
        return pd.DataFrame()


def load_kr_export_growth() -> pd.DataFrame:
    """korea_monthly_trade_data 테이블에서 수출 증감률 계산"""
    try:
        conn = get_conn()
        with conn.cursor() as cur:
            cur.execute(f"""
                SELECT date, root_hs_code AS hs_code, value
                FROM {TBL_KR_TRADE}
                WHERE indicator='expDlr'
                ORDER BY root_hs_code, date
            """)
            rows = cur.fetchall()
        conn.close()
        df = pd.DataFrame(rows)
        if df.empty:
            return pd.DataFrame()
        df['date'] = pd.to_datetime(df['date'])
        records = []
        for code in df['hs_code'].unique():
            sub = df[df['hs_code']==code].sort_values('date')
            vals = sub['value'].tolist()
            if len(vals) < 13:
                continue
            # YoY 증가율 (최신 12개월 vs 전년 12개월)
            cur12  = sum(vals[-12:])
            prev12 = sum(vals[-24:-12]) if len(vals) >= 24 else None
            yoy    = (cur12 / prev12 - 1) * 100 if prev12 else None
            records.append({'hs_code': str(code)[:6], 'kr_yoy': yoy, 'cur12m': cur12})
        return pd.DataFrame(records)
    except Exception as e:
        print(f'[WARN] 한국 수출 데이터 로드 실패: {e}')
        return pd.DataFrame()


df_us_exp = load_us_export_growth()
df_kr_exp = load_kr_export_growth()

# HS Code → 섹터 매핑
def map_sector(hs_code: str) -> str:
    return HS_SECTOR_MAP.get(str(hs_code)[:6], 'Unknown')

# 수혜 업종 집계
beneficiary_sectors = set()

if not df_us_exp.empty:
    df_us_exp['sector'] = df_us_exp['hs_code'].apply(map_sector)
    top_exp = df_us_exp.dropna(subset=['growth_4q']).sort_values('growth_4q', ascending=False).head(30)
    beneficiary_sectors.update(top_exp[top_exp['sector']!='Unknown']['sector'].tolist())
    print('\n[미국 수출 성장 상위 HS Code]')
    display(top_exp[['hs_code','sector','growth_4q','recent_avg']].head(15))

if not df_kr_exp.empty:
    df_kr_exp['sector'] = df_kr_exp['hs_code'].apply(map_sector)
    top_kr = df_kr_exp.dropna(subset=['kr_yoy']).sort_values('kr_yoy', ascending=False).head(30)
    beneficiary_sectors.update(top_kr[top_kr['sector']!='Unknown']['sector'].tolist())
    print('\n[한국 수출 YoY 성장 상위 HS Code]')
    display(top_kr[['hs_code','sector','kr_yoy','cur12m']].head(15))

# 에너지는 제외 업종이므로 필터
beneficiary_sectors.discard('Energy')
beneficiary_sectors.discard('Unknown')

print(f'\n[수혜 예상 업종]: {sorted(beneficiary_sectors)}')

## Cell 7 · 매출 고성장 기업 추출

In [ ]:
# ── 4Q 매출 성장률 기준 고성장 기업 ──────────────────────────
if 'growth_4q' in df_val.columns:
    df_high_growth = df_val[
        df_val['growth_4q'].notna() &
        (df_val['growth_4q'] >= GROWTH_4Q_THRESH)
    ].copy()
else:
    df_high_growth = df_val.copy()
    print('[WARN] growth_4q 컬럼 없음 — 전체 사용')

df_high_growth = df_high_growth.sort_values('growth_4q', ascending=False)
high_growth_tickers = set(df_high_growth['ticker'].tolist())

print(f'[매출 고성장 기업] ({GROWTH_4Q_THRESH}% 이상) : {len(high_growth_tickers)}개')
display(df_high_growth[['ticker','sector','growth_4q','growth_8q','current_price']].head(30))

## Cell 8 · 교집합 필터 + AI Pick

In [ ]:
# ──────────────────────────────────────────────────────────────
# 8-1. 수혜 업종 × 매출 고성장 교집합
# ──────────────────────────────────────────────────────────────
df_candidate = df_val[
    df_val['ticker'].isin(high_growth_tickers)
].copy()

if beneficiary_sectors:
    in_sector = df_candidate['sector'].isin(beneficiary_sectors)
    df_intersect = df_candidate[in_sector].copy()
    df_only_growth = df_candidate[~in_sector].copy()
    print(f'[교집합] 수혜업종 × 매출고성장 : {len(df_intersect)}개')
    print(f'[매출고성장만 (수혜업종 제외)] : {len(df_only_growth)}개')
else:
    df_intersect   = df_candidate.copy()
    df_only_growth = pd.DataFrame()
    print(f'[수혜업종 데이터 없음] 고성장 기업 전체 사용: {len(df_intersect)}개')

display(df_intersect[['ticker','sector','growth_4q','upside_weighted']].head(30))

# ──────────────────────────────────────────────────────────────
# 8-2. AI Pick (Dr.Know 추천)
# 매출 고성장 기업 중 AI 지식 기반으로 성장성 유망 기업 추천
# ──────────────────────────────────────────────────────────────
AI_PICKS = {
    # 반도체/AI 인프라
    'NVDA': 'AI GPU 수요 급증, 데이터센터 확장 지속',
    'AVGO': 'AI ASIC 설계 + 네트워킹 솔루션 독점적 지위',
    'ANET': '하이퍼스케일 AI 클러스터 네트워킹 급성장',
    'LRCX': '차세대 메모리(HBM) 식각 장비 수혜',
    'KLAC': '반도체 검사장비 - AI 고도화에 필수',
    # 소프트웨어/클라우드
    'MSFT': 'Azure + Copilot AI 수익화 가속',
    'CRM':  'AI 기반 Agentforce - 엔터프라이즈 AI 선점',
    'NOW':  'AI 워크플로우 자동화 급성장',
    'PANW': 'AI 보안 플랫폼 통합화 - 고객 확보 가속',
    # 헬스케어
    'LLY':  'GLP-1(오젬픽 계열) 비만치료제 시장 선도',
    'VRTX': '낭포성 섬유증 + 비알콜성 간질환 파이프라인',
    # 방산/우주
    'AXON': '공공안전 AI + 테이저 확장 - 비방산 방산주',
    'HEI':  '항공 MRO 부품 독점 공급자',
    # 소비재
    'CPRT': '자동차 경매 온라인화 - 구조적 성장',
    'FTNT': 'AI 네트워크 보안 통합 플랫폼',
}

# AI Pick 중 고성장 기업과 겹치는 것 강조
ai_in_candidate = {t: r for t, r in AI_PICKS.items() if t in high_growth_tickers}

print(f'\n[AI Pick] 매출 고성장 교집합: {len(ai_in_candidate)}개')
for tkr, reason in ai_in_candidate.items():
    info = US_SECTOR_MAP.get(tkr, {})
    print(f'  {tkr:<8} [{info.get("sector","N/A"):20}] {reason}')

## Cell 9 · 가중 적정주가 & Upside 순위 (매수 후보 20~50개)

In [ ]:
# ──────────────────────────────────────────────────────────────
# 교집합 + AI Pick 합집합 → Upside 기준 필터 → 최종 후보
# ──────────────────────────────────────────────────────────────
candidate_tickers = set(df_intersect['ticker'].tolist()) | set(AI_PICKS.keys())

# Upside 필터 적용
df_screen = df_val[df_val['ticker'].isin(candidate_tickers)].copy()

def upside_threshold(row):
    cap = row.get('mktCap_B') or 0
    return UPSIDE_SMALL_CAP if float(cap) <= SMALL_CAP_THRESH else UPSIDE_LARGE_CAP

df_screen['upside_min'] = df_screen.apply(upside_threshold, axis=1)

if 'upside_weighted' in df_screen.columns:
    df_screen = df_screen[
        df_screen['upside_weighted'].notna() &
        (df_screen['upside_weighted'] / 100 >= df_screen['upside_min'])
    ].copy()

# AI Pick 여부 표시
df_screen['ai_pick']    = df_screen['ticker'].map(lambda t: AI_PICKS.get(t, ''))
df_screen['ai_flag']    = df_screen['ai_pick'].apply(lambda x: '★' if x else '')
df_screen['in_sector']  = df_screen['sector'].isin(beneficiary_sectors)

# Upside 내림차순 정렬 + 상위 50개
sort_col = 'upside_weighted' if 'upside_weighted' in df_screen.columns else df_screen.columns[0]
df_final = df_screen.sort_values(sort_col, ascending=False).head(50)

# 출력 컬럼
show_cols = ['ticker','sector','mktCap_B','current_price',
             'tp_fcff','tp_relv','tp_weighted','upside_weighted',
             'growth_4q','in_sector','ai_flag']
show_cols = [c for c in show_cols if c in df_final.columns]

print('=' * 80)
print(f'  최종 매수 후보 ({len(df_final)}개) — Upside 순서')
print(f'  가중치: FCFF {W_FCFF*100:.0f}% + Relative {W_RELV*100:.0f}%')
print(f'  최소 Upside: 대형주 {UPSIDE_LARGE_CAP*100:.0f}% / 중소형주 {UPSIDE_SMALL_CAP*100:.0f}%')
print('=' * 80)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.1f}'.format)
display(df_final[show_cols])

# 간단 시각화
if len(df_final) > 0 and 'upside_weighted' in df_final.columns:
    fig, ax = plt.subplots(figsize=(14, max(6, len(df_final)*0.35)))
    colors = ['#E74C3C' if row['ai_flag'] == '★' else '#3498DB'
              for _, row in df_final.iterrows()]
    bars = ax.barh(df_final['ticker'], df_final['upside_weighted'], color=colors)
    ax.axvline(30, color='gray', linestyle='--', linewidth=1, label='30% 기준선')
    ax.axvline(40, color='orange', linestyle='--', linewidth=1, label='40% 기준선')
    ax.set_xlabel('Weighted Upside (%)')
    ax.set_title(f'매수 후보 Upside 순위 ({EVAL_DATE})')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## Cell 10 · 수출입 Top-20 HS Code 전월/당월 예측 변화

In [ ]:
# ──────────────────────────────────────────────────────────────
# 미국 수출 예측 상위 20개 HS Code: 전월/당월 예측 변화 비교
# ──────────────────────────────────────────────────────────────
def load_export_top20_change() -> pd.DataFrame:
    """
    us_trade_export_monthly_with_forecast에서
    - 최신 created_at (당월 예측)
    - 직전 created_at (전월 예측)
    의 향후 12개월 합산을 비교해서 상위 20개 HS Code 변화율 반환
    """
    try:
        conn = get_conn()
        with conn.cursor() as cur:
            # 최신 2개 created_at 추출
            cur.execute(f"""
                SELECT DISTINCT created_at
                FROM {TBL_US_EXP_M}
                WHERE is_forecast=1
                ORDER BY created_at DESC
                LIMIT 2
            """)
            dates = [r['created_at'] for r in cur.fetchall()]
        conn.close()

        if len(dates) < 2:
            print('[WARN] 비교할 예측 날짜가 2개 미만')
            return pd.DataFrame()

        curr_date, prev_date = dates[0], dates[1]
        print(f'  당월 예측: {curr_date}')
        print(f'  전월 예측: {prev_date}')

        def get_fcst_sum(created_at) -> Dict:
            conn2 = get_conn()
            with conn2.cursor() as cur:
                cur.execute(f"""
                    SELECT hs_code, SUM(expDlr_forecast) AS total_fcst
                    FROM {TBL_US_EXP_M}
                    WHERE is_forecast=1 AND created_at='{created_at}'
                    GROUP BY hs_code
                """)
                rows = cur.fetchall()
            conn2.close()
            return {r['hs_code']: r['total_fcst'] for r in rows}

        curr_sum = get_fcst_sum(curr_date)
        prev_sum = get_fcst_sum(prev_date)

        records = []
        for hs, curr_val in curr_sum.items():
            prev_val = prev_sum.get(hs)
            if prev_val and prev_val > 0:
                chg = (curr_val - prev_val) / prev_val * 100
            else:
                chg = None
            records.append({
                'hs_code': str(hs)[:6],
                'sector' : HS_SECTOR_MAP.get(str(hs)[:6], 'Unknown'),
                f'fcst_{str(prev_date)[:7]}': prev_val,
                f'fcst_{str(curr_date)[:7]}': curr_val,
                'chg_pct': chg,
            })

        df = pd.DataFrame(records).dropna(subset=['chg_pct'])
        # 전체 합산 규모 기준 상위 20개
        fcst_col = f'fcst_{str(curr_date)[:7]}'
        return df.sort_values(fcst_col, ascending=False).head(20)

    except Exception as e:
        print(f'[WARN] 수출 변화 분석 실패: {e}')
        return pd.DataFrame()


df_exp_change = load_export_top20_change()

if not df_exp_change.empty:
    print('\n[수출 상위 20 HS Code — 전월/당월 예측 변화]')
    display(df_exp_change)

    # 시각화
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['#E74C3C' if v > 0 else '#3498DB'
              for v in df_exp_change['chg_pct']]
    ax.bar(df_exp_change['hs_code'], df_exp_change['chg_pct'], color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('HS Code (6자리)')
    ax.set_ylabel('전월 대비 예측 변화 (%)')
    ax.set_title('미국 수출 Top-20 HS Code 예측 변화 (전월 vs 당월)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] 수출 변화 데이터 없음 (테이블 비어있거나 예측 날짜 부족)')

## Cell 11 · Excel 보고서 저장

In [ ]:
# ──────────────────────────────────────────────────────────────
# Excel 보고서: 시트별 구성
#  1) 매수후보_랭킹   — 최종 매수 후보 Upside 순서
#  2) 전체_Valuation  — 1026개 전체 valuation 결과
#  3) 매출고성장      — growth_4q 기준 고성장 기업
#  4) 수출입_수혜업종  — HS Code 수출입 성장 + 섹터 매핑
#  5) FCFF_상세       — FCFF 계산 근거 (Re, ROIC, WACC, FCFF 등)
#  6) Relative_상세   — PBR/PSR/PER 이론값 계산 근거
#  7) Top20_수출변화  — 수출 예측 전월/당월 변화
# ──────────────────────────────────────────────────────────────
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils.dataframe import dataframe_to_rows
    OPENPYXL_OK = True
except ImportError:
    print('[WARN] openpyxl 없음. pip install openpyxl')
    OPENPYXL_OK = False

if OPENPYXL_OK:
    writer = pd.ExcelWriter(REPORT_FILE, engine='openpyxl')

    def _write_sheet(df: pd.DataFrame, sheet: str, float_fmt: str = '{:.2f}'):
        if df.empty:
            pd.DataFrame({'메시지':['데이터 없음']}).to_excel(writer, sheet_name=sheet, index=False)
            return
        df.to_excel(writer, sheet_name=sheet, index=False)

    # 1) 매수후보 랭킹
    _write_sheet(df_final, '매수후보_랭킹')

    # 2) 전체 Valuation
    _write_sheet(df_val, '전체_Valuation')

    # 3) 매출 고성장
    _write_sheet(df_high_growth, '매출고성장')

    # 4) 수출입 수혜
    if not df_us_exp.empty:
        _write_sheet(df_us_exp.sort_values('growth_4q', ascending=False), '미국수출_수혜')
    if not df_kr_exp.empty:
        _write_sheet(df_kr_exp.sort_values('kr_yoy', ascending=False), '한국수출_수혜')

    # 5) FCFF 상세
    fcff_detail_cols = [
        'ticker','date_fcff','discount_rate','roic','g_terminal',
        'opm_forecast','tax_rate','reinvestment_rate','fcff',
        'enterprise_value','net_debt','equity_value','shares',
        'tp_fcff','current_price','upside_fcff'
    ]
    fcff_detail_cols = [c for c in fcff_detail_cols if c in df_val.columns]
    _write_sheet(df_val[fcff_detail_cols].dropna(subset=['tp_fcff']), 'FCFF_상세')

    # 6) Relative 상세
    relv_detail_cols = [
        'ticker','sector','roe_y1','roe_y2','g_est','re_mid','beta_ensemble',
        'pbr_theory','psr_theory','per_theory',
        'tp_pbr','tp_psr','tp_per','tp_relv','current_price','upside_relv'
    ]
    relv_detail_cols = [c for c in relv_detail_cols if c in df_val.columns]
    _write_sheet(df_val[relv_detail_cols].dropna(subset=['tp_relv']), 'Relative_상세')

    # 7) Top20 수출 변화
    if not df_exp_change.empty:
        _write_sheet(df_exp_change, 'Top20_수출변화')

    writer.close()
    print(f'[OK] Excel 보고서 저장 완료: {REPORT_FILE}')
else:
    print('[SKIP] openpyxl 미설치로 Excel 저장 생략')